# Chapter 2 - The Simple Regression Model

**Introductory Econometrics** by Jeffrey M. Wooldridge (5th Edition)

---

This notebook contains solutions to computer exercises from Chapter 2. These exercises focus on simple linear regression, OLS estimation, and interpreting regression results.

## Setup

Import necessary libraries and load utility functions.

In [1]:
# Standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Custom utilities
import sys
sys.path.append('..')
from utils import (
    load_wooldridge_data,
    summary_statistics,
    plot_correlation_matrix
)

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 4)
%matplotlib inline

print("Setup complete!")

Setup complete!


---

## Computer Exercise C1: 401(k) Pension Plans

**Problem Statement:**

The data in 401K.RAW are a subset of data analyzed by Papke (1995) to study the relationship between participation in a 401(k) pension plan and the generosity of the plan. The variable prate is the percentage of eligible workers with an active account; this is the variable we would like to explain. The measure of generosity is the plan match rate, mrate. This variable gives the average amount the firm contributes to each worker's plan for each $1 contribution by the worker.

(i) Find the average participation rate and the average match rate in the sample of plans.

(ii) Now, estimate the simple regression equation prate = β₀ + β₁mrate, and report the results along with the sample size and R-squared.

(iii) Interpret the intercept in your equation. Interpret the coefficient on mrate.

(iv) Find the predicted prate when mrate = 3.5. Is this a reasonable prediction? Explain what is happening here.

(v) How much of the variation in prate is explained by mrate? Is this a lot in your opinion?

**Dataset:** 401k

### Data Loading and Initial Exploration

In [79]:
# Load the 401K dataset
k401 = load_wooldridge_data('401k')

print("401K Dataset Overview:")
print(f"Shape: {k401.shape}")
print(f"Columns: {list(k401.columns)}")
print("\nFirst 5 rows:")
print(k401.head())

CSV file not found for 401k. Downloading...
Successfully downloaded and converted to: d:\Users\achyu\git\learnStats\Econometrics\Practice\notebooks\..\data\401k.csv
Loaded 401k from d:\Users\achyu\git\learnStats\Econometrics\Practice\notebooks\..\data\401k.csv
Shape: (1534, 8)
401K Dataset Overview:
Shape: (1534, 8)
Columns: ['prate', 'mrate', 'totpart', 'totelg', 'age', 'totemp', 'sole', 'ltotemp']

First 5 rows:
   prate  mrate  totpart  totelg   age  totemp   sole  ltotemp
0   26.1   0.21   1653.0  6322.0   8.0  8709.0  False   9.0721
1  100.0   1.42    262.0   262.0   6.0   315.0   True   5.7526
2   97.6   0.91    166.0   170.0  10.0   275.0   True   5.6168
3  100.0   0.42    257.0   257.0   7.0   500.0  False   6.2146
4   82.5   0.53    591.0   716.0  28.0   933.0   True   6.8384
Successfully downloaded and converted to: d:\Users\achyu\git\learnStats\Econometrics\Practice\notebooks\..\data\401k.csv
Loaded 401k from d:\Users\achyu\git\learnStats\Econometrics\Practice\notebooks\..\d

### Part (i): Sample Averages

In [81]:
# Part (i): Find average participation rate and average match rate
avg_participation = k401['prate'].mean()
avg_match_rate = k401['mrate'].mean()
print(f"Average Participation Rate: {avg_participation:.4f}")
print(f"Average Match Rate: {avg_match_rate:.4f}")

Average Participation Rate: 87.3629
Average Match Rate: 0.7315


### Part (ii): Simple Regression Estimation

In [85]:
# Part (ii): Estimate prate = β₀ + β₁mrate
import statsmodels.api as sm
X = sm.add_constant(k401['mrate'])
y = k401['prate']
model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                  prate   R-squared:                       0.075
Model:                            OLS   Adj. R-squared:                  0.074
Method:                 Least Squares   F-statistic:                     123.7
Date:                Tue, 04 Nov 2025   Prob (F-statistic):           1.10e-27
Time:                        14:11:28   Log-Likelihood:                -6437.0
No. Observations:                1534   AIC:                         1.288e+04
Df Residuals:                    1532   BIC:                         1.289e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         83.0755      0.563    147.484      0.0

### Part (iii): Interpretation of Coefficients

The intercept at around 83% shows the expected participation rate at a 401k match rate of 0%. The coefficient 5.86 indicates that for every unit increase in match rage we see a unit increase of participation by 5.86%. 

### Part (iv): Prediction for mrate = 3.5

In [88]:
# Part (iv): Predict prate when mrate = 3.5

# Extract coefficients from the regression model
intercept = model.params[0]  # β₀
slope = model.params[1]      # β₁

# Simply plug into the estimated equation: prate = β₀ + β₁ * mrate
mrate_value = 3.5
predicted_prate = intercept + slope * mrate_value

print(f"Estimated equation: prate = {intercept:.4f} + {slope:.4f} * mrate")
print(f"When mrate = {mrate_value}:")
print(f"Predicted prate = {intercept:.4f} + {slope:.4f} * {mrate_value} = {predicted_prate:.4f}")

# Check if this is reasonable
print(f"\nIs this reasonable?")
print(f"- This predicts a participation rate of {predicted_prate:.1f}%")
print(f"- Sample average participation rate: {k401['prate'].mean():.1f}%")
print(f"- Sample range: {k401['prate'].min():.1f}% to {k401['prate'].max():.1f}%")

if predicted_prate > 100:
    print("- WARNING: Prediction exceeds 100%! This suggests extrapolation beyond reasonable range.")
elif predicted_prate < 0:
    print("- WARNING: Prediction is negative! This suggests extrapolation beyond reasonable range.")
else:
    print("- This prediction seems reasonable as it falls within a plausible range.")

Estimated equation: prate = 83.0755 + 5.8611 * mrate
When mrate = 3.5:
Predicted prate = 83.0755 + 5.8611 * 3.5 = 103.5892

Is this reasonable?
- This predicts a participation rate of 103.6%
- Sample average participation rate: 87.4%
- Sample range: 3.0% to 100.0%
- WARNING: Prediction exceeds 100%! This suggests extrapolation beyond reasonable range.


C:\Users\achyu\AppData\Local\Temp\ipykernel_29464\1199461320.py:4: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept = model.params[0]  # β₀
C:\Users\achyu\AppData\Local\Temp\ipykernel_29464\1199461320.py:5: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  slope = model.params[1]      # β₁


I think this is an unreasonable prediction here. Percentage relationships are usually not well captured by standard linear models. I believe using log or log log models would be necessary here to show percentage increases within variables.

### Part (v): Goodness of Fit Assessment

In [89]:
# Part (v): Assess how much variation is explained by mrate
r_squared = model.rsquared
print(f"\nR-squared: {r_squared:.4f}")


R-squared: 0.0747


This significantly small r squared value shows that the match rate does not explain much of the participation rate's variance

---

## Computer Exercise C2: CEO Salary and Tenure

**Problem Statement:**

The data set in CEOSAL2.RAW contains information on chief executive officers for U.S. corporations. The variable salary is annual compensation, in thousands of dollars, and ceoten is prior number of years as company CEO.

(i) Find the average salary and the average tenure in the sample.

(ii) How many CEOs are in their first year as CEO (that is, ceoten = 0)? What is the longest tenure as a CEO?

(iii) Estimate the simple regression model log(salary) = β₀ + β₁ceoten + u, and report your results in the usual form. What is the (approximate) predicted percentage increase in salary given one more year as a CEO?

**Dataset:** ceosal2

### Data Loading and Initial Exploration

In [92]:
# Load the CEOSAL2 dataset
ceosal2 = load_wooldridge_data('ceosal2')

print("CEOSAL2 Dataset Overview:")
print(f"Shape: {ceosal2.shape}")
print(f"Columns: {list(ceosal2.columns)}")
print("\nFirst 5 rows:")
print(ceosal2.head())

CSV file not found for ceosal2. Downloading...
Successfully downloaded and converted to: d:\Users\achyu\git\learnStats\Econometrics\Practice\notebooks\..\data\ceosal2.csv
Loaded ceosal2 from d:\Users\achyu\git\learnStats\Econometrics\Practice\notebooks\..\data\ceosal2.csv
Shape: (177, 15)
CEOSAL2 Dataset Overview:
Shape: (177, 15)
Columns: ['salary', 'age', 'college', 'grad', 'comten', 'ceoten', 'sales', 'profits', 'mktval', 'lsalary', 'lsales', 'lmktval', 'comtensq', 'ceotensq', 'profmarg']

First 5 rows:
   salary   age  college   grad  comten  ceoten   sales  profits   mktval  \
0  1161.0  49.0     True   True     9.0     2.0  6200.0    966.0  23200.0   
1   600.0  43.0     True   True    10.0    10.0   283.0     48.0   1100.0   
2   379.0  51.0     True   True     9.0     3.0   169.0     40.0   1100.0   
3   651.0  55.0     True  False    22.0    22.0  1100.0    -54.0   1000.0   
4   497.0  44.0     True   True     8.0     6.0   351.0     28.0    387.0   

   lsalary  lsales  lmktv

### Part (i): Sample Statistics

In [94]:
# Part (i): Find average salary and average tenure
avg_salary = ceosal2['salary'].mean()
avg_tenure = ceosal2['comten'].mean()
print(f"Average Salary: {avg_salary:.2f}")
print(f"Average Tenure: {avg_tenure:.2f}")

Average Salary: 865.86
Average Tenure: 22.50


### Part (ii): Tenure Distribution

In [95]:
# Part (ii): Count first-year CEOs and find maximum tenure
first_year_ceos = ceosal2[ceosal2['comten'] == 0].shape[0]
max_tenure = ceosal2['comten'].max()
print(f"Number of first-year CEOs: {first_year_ceos}")
print(f"Maximum Tenure: {max_tenure}")


Number of first-year CEOs: 0
Maximum Tenure: 58.0


### Part (iii): Log-Linear Regression

In [102]:
# Part (iii): Estimate log(salary) = β₀ + β₁ceoten + u

# Method 1: Manual log transformation + OLS (Standard approach for log-linear models)
X = sm.add_constant(ceosal2['ceoten']) 
y = np.log(ceosal2['salary'])  # Log-transform the dependent variable
model_manual = sm.OLS(y, X).fit()

print("=== MANUAL LOG TRANSFORMATION + OLS ===")
print(model_manual.summary())

# Extract coefficients for interpretation
beta_0 = model_manual.params[0]
beta_1 = model_manual.params[1]

print(f"\n=== INTERPRETATION ===")
print(f"Estimated equation: log(salary) = {beta_0:.4f} + {beta_1:.4f} * ceoten")
print(f"Coefficient interpretation:")
print(f"- β₁ = {beta_1:.4f} means a 1-year increase in CEO tenure increases salary by approximately {beta_1*100:.2f}%")
print(f"- More precisely: %Δsalary ≈ 100 × β₁ × Δceoten for small changes")

=== MANUAL LOG TRANSFORMATION + OLS ===
                            OLS Regression Results                            
Dep. Variable:                 salary   R-squared:                       0.013
Model:                            OLS   Adj. R-squared:                  0.008
Method:                 Least Squares   F-statistic:                     2.334
Date:                Tue, 04 Nov 2025   Prob (F-statistic):              0.128
Time:                        15:11:17   Log-Likelihood:                -160.84
No. Observations:                 177   AIC:                             325.7
Df Residuals:                     175   BIC:                             332.0
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       

C:\Users\achyu\AppData\Local\Temp\ipykernel_29464\3347297467.py:12: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  beta_0 = model_manual.params[0]
C:\Users\achyu\AppData\Local\Temp\ipykernel_29464\3347297467.py:13: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  beta_1 = model_manual.params[1]


---

## Computer Exercise C3: Sleep vs Work Tradeoff

**Problem Statement:**

Use the data in SLEEP75.RAW from Biddle and Hamermesh (1990) to study whether there is a tradeoff between the time spent sleeping per week and the time spent in paid work. We could use either variable as the dependent variable. For concreteness, estimate the model sleep = β₀ + β₁totwrk + u, where sleep is minutes spent sleeping at night per week and totwrk is total minutes worked during the week.

(i) Report your results in equation form along with the number of observations and R². What does the intercept in this equation mean?

(ii) If totwrk increases by 2 hours, by how much is sleep estimated to fall? Do you find this to be a large effect?

**Dataset:** sleep75

### Data Loading and Initial Exploration

In [109]:
# Load the SLEEP75 dataset
sleep75 = load_wooldridge_data('sleep75')

print("SLEEP75 Dataset Overview:")
print(f"Shape: {sleep75.shape}")
print(f"Columns: {list(sleep75.columns)}")
print("\nFirst 5 rows:")
print(sleep75.head())

CSV file not found for sleep75. Downloading...
Error processing sleep75: Need to pass bool-like values


FileNotFoundError: Could not download sleep75

### Part (i): Sleep-Work Regression

In [ ]:
# Part (i): Estimate sleep = β₀ + β₁totwrk + u

### Part (ii): Effect of 2-Hour Work Increase

In [ ]:
# Part (ii): Calculate effect of 2-hour increase in work time

---

## Computer Exercise C4: IQ and Wages

**Problem Statement:**

Use the data in WAGE2.RAW to estimate a simple regression explaining monthly salary (wage) in terms of IQ score (IQ).

(i) Find the average salary and average IQ in the sample. What is the sample standard deviation of IQ? (IQ scores are standardized so that the average in the population is 100 with a standard deviation equal to 15.)

(ii) Estimate a simple regression model where a one-point increase in IQ changes wage by a constant dollar amount. Use this model to find the predicted increase in wage for an increase in IQ of 15 points. Does IQ explain most of the variation in wage?

(iii) Now, estimate a model where each one-point increase in IQ has the same percentage effect on wage. If IQ increases by 15 points, what is the approximate percentage increase in predicted wage?

**Dataset:** wage2

### Data Loading and Initial Exploration

In [ ]:
# Load the WAGE2 dataset
wage2 = load_wooldridge_data('wage2')

print("WAGE2 Dataset Overview:")
print(f"Shape: {wage2.shape}")
print(f"Columns: {list(wage2.columns)}")
print("\nFirst 5 rows:")
print(wage2.head())

### Part (i): Sample Statistics

In [ ]:
# Part (i): Find average salary, average IQ, and standard deviation of IQ

### Part (ii): Linear Model (Level-Level)

In [ ]:
# Part (ii): Estimate wage = β₀ + β₁IQ + u (linear model)

### Part (iii): Log-Linear Model (Log-Level)

In [ ]:
# Part (iii): Estimate log(wage) = β₀ + β₁IQ + u (log-linear model)

---

## Computer Exercise C5: R&D and Sales Elasticity

**Problem Statement:**

For the population of firms in the chemical industry, let rd denote annual expenditures on research and development, and let sales denote annual sales (both are in millions of dollars).

(i) Write down a model (not an estimated equation) that implies a constant elasticity between rd and sales. Which parameter is the elasticity?

(ii) Now, estimate the model using the data in RDCHEM.RAW. Write out the estimated equation in the usual form. What is the estimated elasticity of rd with respect to sales? Explain in words what this elasticity means.

**Dataset:** rdchem

### Data Loading and Initial Exploration

In [ ]:
# Load the RDCHEM dataset
rdchem = load_wooldridge_data('rdchem')

print("RDCHEM Dataset Overview:")
print(f"Shape: {rdchem.shape}")
print(f"Columns: {list(rdchem.columns)}")
print("\nFirst 5 rows:")
print(rdchem.head())

### Part (i): Constant Elasticity Model

In [ ]:
# Part (i): Discuss the constant elasticity model form

### Part (ii): Log-Log Regression

In [ ]:
# Part (ii): Estimate log(rd) = β₀ + β₁log(sales) + u

---

## Computer Exercise C6: Math Scores and School Spending

**Problem Statement:**

We used the data in MEAP93.RAW for Example 2.12. Now we want to explore the relationship between the math pass rate (math10) and spending per student (expend).

(i) Do you think each additional dollar spent has the same effect on the pass rate, or does a diminishing effect seem more appropriate? Explain.

(ii) In the population model math10 = β₀ + β₁log(expend) + u, argue that β₁/10 is the percentage point change in math10 given a 10% increase in expend.

(iii) Use the data in MEAP93.RAW to estimate the model from part (ii). Report the estimated equation in the usual way, including the sample size and R-squared.

(iv) How big is the estimated spending effect? Namely, if spending increases by 10%, what is the estimated percentage point increase in math10?

(v) One might worry that regression analysis can produce fitted values for math10 that are greater than 100. Why is this not much of a worry in this data set?

**Dataset:** meap93

### Data Loading and Initial Exploration

In [ ]:
# Load the MEAP93 dataset
meap93 = load_wooldridge_data('meap93')

print("MEAP93 Dataset Overview:")
print(f"Shape: {meap93.shape}")
print(f"Columns: {list(meap93.columns)}")
print("\nFirst 5 rows:")
print(meap93.head())

### Part (i): Functional Form Discussion

In [ ]:
# Part (i): Discuss linear vs diminishing effects of spending

### Part (ii): Semi-Log Model Interpretation

In [ ]:
# Part (ii): Explain the β₁/10 interpretation

### Part (iii): Semi-Log Regression

In [ ]:
# Part (iii): Estimate math10 = β₀ + β₁log(expend) + u

### Part (iv): Spending Effect Calculation

In [ ]:
# Part (iv): Calculate the effect of a 10% increase in spending

### Part (v): Fitted Values Discussion

In [ ]:
# Part (v): Check range of fitted values and discuss bounds

---

## Computer Exercise C7: Charitable Giving

**Problem Statement:**

Use the data in CHARITY.RAW [obtained from Franses and Paap (2001)] to answer the following questions:

(i) What is the average gift in the sample of 4,268 people (in Dutch guilders)? What percentage of people gave no gift?

(ii) What is the average mailings per year? What are the minimum and maximum values?

(iii) Estimate the model gift = β₀ + β₁mailsyear + u by OLS and report the results in the usual way, including the sample size and R-squared.

(iv) Interpret the slope coefficient. If each mailing costs one guilder, is the charity expected to make a net gain on each mailing? Does this mean the charity makes a net gain on every mailing? Explain.

(v) What is the smallest predicted charitable contribution in the sample? Using this simple regression analysis, can you ever predict zero for gift?

**Dataset:** charity

### Data Loading and Initial Exploration

In [ ]:
# Load the CHARITY dataset
charity = load_wooldridge_data('charity')

print("CHARITY Dataset Overview:")
print(f"Shape: {charity.shape}")
print(f"Columns: {list(charity.columns)}")
print("\nFirst 5 rows:")
print(charity.head())

### Part (i): Gift Statistics

In [ ]:
# Part (i): Average gift and percentage giving no gift

### Part (ii): Mailings Statistics

In [ ]:
# Part (ii): Average, minimum, and maximum mailings per year

### Part (iii): Gift-Mailings Regression

In [ ]:
# Part (iii): Estimate gift = β₀ + β₁mailsyear + u

### Part (iv): Cost-Benefit Analysis

In [ ]:
# Part (iv): Interpret slope and assess net gain from mailings

### Part (v): Prediction Analysis

In [ ]:
# Part (v): Find smallest predicted contribution and discuss zero predictions

---

## Computer Exercise C8: Monte Carlo Simulation

**Problem Statement:**

To complete this exercise you need a software package that allows you to generate data from the uniform and normal distributions.


**Dataset:** Simulated data

### Part (i): Generate Explanatory Variable

(i) Start by generating 500 observations xᵢ – the explanatory variable – from the uniform distribution with range [0,10]. What are the sample mean and sample standard deviation of the xᵢ?

In [37]:
np.random.seed(42)  # For reproducibility

# Part (i): Generate 500 observations from Uniform[0,10]
x = np.random.uniform(low = 0, high = 10, size = 500)
print(x.mean())
print(x.std(ddof = 0))
print(x.std(ddof=1)) # ddof means degrees of freedom


4.985617122340139
2.9838957050944543
2.9868840841159234


I am using ddof = 1 as this will change the degrees of freedom to get the sample standard deviation instead of population standard deviation

### Part (ii): Generate Error Terms
(ii) Randomly generate 500 errors, uᵢ, from the Normal[0,36] distribution. Is the sample average of the uᵢ exactly zero? Why or why not? What is the sample standard deviation of the uᵢ?

If you generate a Normal[0,1], as is commonly available, simply multiply the outcomes by six.

In [41]:
# Part (ii): Generate 500 errors from Normal[0,36]


# Normal[0,36] means mean=0, variance=36, so standard deviation = √36 = 6

np.random.seed(42)  # For reproducibility

# Method 1: Direct generation with correct standard deviation
errors = np.random.normal(loc=0, scale=6, size=500)

# Method 2: Following textbook hint - generate Normal[0,1] then multiply by 6
# errors_std = np.random.standard_normal(size=500)
# errors = 6 * errors_std

print(f"Sample mean of errors: {errors.mean():.6f}")
print(f"Sample standard deviation: {errors.std(ddof=1):.4f}")
print(f"Sample variance: {errors.var(ddof=1):.4f}")
print(f"\nExpected values:")
print(f"True mean: 0")
print(f"True standard deviation: 6") 
print(f"True variance: 36")

Sample mean of errors: 0.041028
Sample standard deviation: 5.8875
Sample variance: 34.6629

Expected values:
True mean: 0
True standard deviation: 6
True variance: 36


The sample average is near zero but not exactly zero as it is random generated.

In [33]:
errors

array([ 12.30321513,  67.54215021,  34.21525817, -20.7685316 ,
       -32.34292817,  17.70909017, -47.52839545,  65.93251557,
        42.45984435, -16.89032348, -61.67284305,  48.73940547,
        -4.12343443,  44.56138723, -57.39939572, -21.57750083,
         0.18877319,   1.69130138, -16.20235697,  22.42259756,
       -38.43433546,  -5.12566146,   4.33064274,  18.51979803,
        25.61813561, -40.48711531, -55.22811015,  45.99636559,
        11.96330443, -26.94551532,  55.84147112,   4.16428683,
        42.45469863,   2.43066533,  74.1869253 ,  63.19227033,
        -8.96270935,  34.97655423,  23.23353419,  49.27073607,
       -34.73724458,  24.69785256,  38.10328153, -63.31462151,
       -42.59730646, -73.4123584 ,  -9.69864604,  25.83152121,
        54.08485388,   2.6674121 ,  58.63015964, -49.6836525 ,
       -61.32176782,  -1.99971716,  13.82635616,  -1.17701093,
       -74.4279156 ,  -3.20832142, -46.96090202,  24.10821176,
        13.19753686, -33.83567231, -18.49920902, -38.13

### Part (iii): Generate Dependent Variable and Run Regression
(iii) Now generate the yᵢ as yᵢ = 1 + 2xᵢ + uᵢ = β₀ + β₁xᵢ + uᵢ; that is, the population intercept is one and the population slope is two. Use the data to run the regression of yᵢ on xᵢ. What are your estimates of the intercept and slope? Are they equal to the population values in the above equation? Explain.

In [44]:
# Part (iii): Generate y = 1 + 2x + u and estimate regression

# Method 1: Using map() and lambda
# y = np.array(list(map(lambda xi, ui: 1 + 2*xi + ui, x, errors)))

# Alternative Method 2: Using list comprehension (more Pythonic)
# y = np.array([1 + 2*xi + ui for xi, ui in zip(x, errors)])

# Alternative Method 3: Vectorized operations (most efficient)
y = 1 + 2*x + errors

print(f"Generated y values with shape: {y.shape}")
print(f"First 5 y values: {y[:5]}")

# Run OLS regression: y = β₀ + β₁x + u
X = sm.add_constant(x)  # Add intercept term
model = sm.OLS(y, X)
results = model.fit()

print("\nOLS Regression Results:")
print(f"Intercept (β₀): {results.params[0]:.4f}")
print(f"Slope (β₁): {results.params[1]:.4f}")
print(f"Population β₀: 1.0000")
print(f"Population β₁: 2.0000")
print(f"\nR-squared: {results.rsquared:.4f}")
print(f"Sample size: {len(y)}")


Generated y values with shape: (500,)
First 5 y values: [11.4710873  19.18470032 19.52601006 22.11134882  2.71545256]

OLS Regression Results:
Intercept (β₀): 0.6523
Slope (β₁): 2.0780
Population β₀: 1.0000
Population β₁: 2.0000

R-squared: 0.5268
Sample size: 500


The intercept here equal to the population values as the population values belong to [0, 10] and beta_0 is in that interval

### Part (iv): Verify OLS Properties with Residuals
(iv) Obtain the OLS residuals, ûᵢ, and verify that equation (2.60) holds (subject to rounding error).

Condition 1: ∑ûᵢ = 0 (sum of residuals equals zero)

Condition 2: ∑xᵢûᵢ = 0 (sum of x times residuals equals zero)

In [49]:
# Part (iv): Check equation (2.60) properties with OLS residuals

# Get OLS residuals
residuals = results.resid

# Equation (2.60): Two OLS first-order conditions

sum_residuals = np.sum(residuals)
print(f"∑ûᵢ = {sum_residuals:.10f}")
print(f"Should be approximately 0 (subject to rounding error)")


sum_x_residuals = np.sum(x * residuals)
print(f"\n∑xᵢûᵢ = {sum_x_residuals:.10f}")
print(f"Should be approximately 0 (subject to rounding error)")


∑ûᵢ = -0.0000000000
Should be approximately 0 (subject to rounding error)

∑xᵢûᵢ = -0.0000000000
Should be approximately 0 (subject to rounding error)


### Part (v): Check Properties with True Errors
(v) Compute the same quantities in equation (2.60) but use the errors uᵢ in place of the residuals. Now what do you conclude?

In [51]:
# Part (v): Check same properties using true errors uᵢ

# Now use the TRUE ERRORS (not OLS residuals)
true_errors = errors

# Check the same two conditions with true errors
sum_true_errors = np.sum(true_errors)
print(f"∑uᵢ = {sum_true_errors:.10f}")
print(f"This is NOT necessarily zero - it's random!")

sum_x_true_errors = np.sum(x * true_errors)
print(f"\n∑xᵢuᵢ = {sum_x_true_errors:.10f}")
print(f"This is also NOT necessarily zero - it's random!")


∑uᵢ = 20.5139837659
This is NOT necessarily zero - it's random!

∑xᵢuᵢ = 449.3624070623
This is also NOT necessarily zero - it's random!


### Part (vi): Repeat with New Sample
(vi) Repeat parts (i), (ii), and (iii) with a new sample of data, starting with generating the xᵢ. Now what do you obtain for β̂₀ and β̂₁? Why are these different from what you obtained in part (iii)?


In [53]:
# Part (vi): Generate new sample and compare estimates
np.random.seed(123)  # Different seed for new sample

# Step 1: Generate new x values from Uniform[0,10]
x_new = np.random.uniform(low=0, high=10, size=500)
print("NEW SAMPLE - Part (i): Explanatory Variable")
print(f"Sample mean of x: {x_new.mean():.4f}")
print(f"Sample standard deviation of x: {x_new.std(ddof=1):.4f}")

# Step 2: Generate new errors from Normal[0,36]
errors_new = np.random.normal(loc=0, scale=6, size=500)
print(f"\nNEW SAMPLE - Part (ii): Error Terms")
print(f"Sample mean of errors: {errors_new.mean():.6f}")
print(f"Sample standard deviation of errors: {errors_new.std(ddof=1):.4f}")
print(f"Sample variance of errors: {errors_new.var(ddof=1):.4f}")

# Step 3: Generate new y values using y = 1 + 2x + u
y_new = 1 + 2*x_new + errors_new

# Step 4: Run OLS regression on new sample
X_new = sm.add_constant(x_new)
model_new = sm.OLS(y_new, X_new)
results_new = model_new.fit()

print(f"\nNEW SAMPLE - Part (iii): OLS Regression Results")
print(f"Intercept (β₀): {results_new.params[0]:.4f}")
print(f"Slope (β₁): {results_new.params[1]:.4f}")
print(f"R-squared: {results_new.rsquared:.4f}")
print(f"Sample size: {len(y_new)}")

print(f"\nCOMPARISON WITH ORIGINAL SAMPLE:")
print(f"Original β₀: {results.params[0]:.4f}, New β₀: {results_new.params[0]:.4f}")
print(f"Original β₁: {results.params[1]:.4f}, New β₁: {results_new.params[1]:.4f}")
print(f"Population β₀: 1.0000, Population β₁: 2.0000")



NEW SAMPLE - Part (i): Explanatory Variable
Sample mean of x: 4.9444
Sample standard deviation of x: 2.8630

NEW SAMPLE - Part (ii): Error Terms
Sample mean of errors: -0.011704
Sample standard deviation of errors: 6.0841
Sample variance of errors: 37.0165

NEW SAMPLE - Part (iii): OLS Regression Results
Intercept (β₀): 0.6532
Slope (β₁): 2.0678
R-squared: 0.4866
Sample size: 500

COMPARISON WITH ORIGINAL SAMPLE:
Original β₀: 0.6523, New β₀: 0.6532
Original β₁: 2.0780, New β₁: 2.0678
Population β₀: 1.0000, Population β₁: 2.0000


Why are these different?
The estimates differ because:
1. We generated a completely new random sample
2. Both x values and error terms are different
3. OLS estimators are random variables that vary across samples
4. Each sample gives different realizations of β̂₀ and β̂₁
5. Both estimates should be close to the true population values (1, 2), but will never be exactly equal due to sampling variation

---

## Problem 3: GPA and ACT Scores

**Problem Statement:**

The following table contains the ACT scores and the GPA (grade point average) for eight college students. Grade point average is based on a four-point scale and has been rounded to one digit after the decimal.

| Student | GPA | ACT |
|---------|-----|-----|
| 1       | 2.8 | 21  |
| 2       | 3.4 | 24  |
| 3       | 3.0 | 26  |
| 4       | 3.5 | 27  |
| 5       | 3.6 | 29  |
| 6       | 3.0 | 25  |
| 7       | 2.7 | 25  |
| 8       | 3.7 | 30  |

(i) Estimate the relationship between GPA and ACT using OLS; that is, obtain the intercept and slope estimates in the equation GPA = β₀ + β₁ACT. Comment on the direction of the relationship. Does the intercept have a useful interpretation here? Explain. How much higher is the GPA predicted to be if the ACT score is increased by five points?

(ii) Compute the fitted values and residuals for each observation, and verify that the residuals (approximately) sum to zero.

(iii) What is the predicted value of GPA when ACT = 20?

(iv) How much of the variation in GPA is explained by ACT? Explain.

### Data Entry

In [54]:
# Enter the GPA and ACT data
gpa_act_data = {
    'Student': [1, 2, 3, 4, 5, 6, 7, 8],
    'GPA': [2.8, 3.4, 3.0, 3.5, 3.6, 3.0, 2.7, 3.7],
    'ACT': [21, 24, 26, 27, 29, 25, 25, 30]
}

gpa_act_df = pd.DataFrame(gpa_act_data)
print("GPA and ACT Data:")
print(gpa_act_df)

GPA and ACT Data:
   Student  GPA  ACT
0        1  2.8   21
1        2  3.4   24
2        3  3.0   26
3        4  3.5   27
4        5  3.6   29
5        6  3.0   25
6        7  2.7   25
7        8  3.7   30


### Part (i): OLS Estimation

In [62]:
# Part (i): Estimate GPA = β₀ + β₁ACT using OLS
X_gpa = sm.add_constant(gpa_act_df['ACT'])
model_gpa = sm.OLS(gpa_act_df['GPA'], X_gpa)
# Print model summary
results_gpa = model_gpa.fit()
print(results_gpa.summary())

                            OLS Regression Results                            
Dep. Variable:                    GPA   R-squared:                       0.577
Model:                            OLS   Adj. R-squared:                  0.507
Method:                 Least Squares   F-statistic:                     8.199
Date:                Tue, 04 Nov 2025   Prob (F-statistic):             0.0287
Time:                        11:47:36   Log-Likelihood:                0.29842
No. Observations:                   8   AIC:                             3.403
Df Residuals:                       6   BIC:                             3.562
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.5681      0.928      0.612      0.5

We see that there exists a positive slope so ACT and GPA are positively correlated. The intercept does not have a useful meaning as one cannot have a score of 0 on the ACT. The intercept here has the purpose of fitting the OLS model. An increase in 5 points in ACT is expected to increase GPA by 0.5 .

### Part (ii): Fitted Values and Residuals

In [63]:
# Part (ii): Compute fitted values and residuals for each observation
fitted_values = results_gpa.fittedvalues
residuals = results_gpa.resid

# Create a detailed table showing all values
gpa_results = gpa_act_df.copy()
gpa_results['Fitted_GPA'] = fitted_values
gpa_results['Residuals'] = residuals

print("Fitted Values and Residuals for each observation:")
print(gpa_results.round(4))

# Verify that residuals sum to approximately zero
residual_sum = residuals.sum()
print(f"\nSum of residuals: {residual_sum:.10f}")
print(f"This should be approximately zero (subject to rounding error)")

# Also show individual fitted values and residuals more clearly
print(f"\nDetailed breakdown:")
for i in range(len(gpa_act_df)):
    print(f"Student {i+1}: GPA={gpa_act_df.iloc[i]['GPA']:.1f}, "
          f"ACT={gpa_act_df.iloc[i]['ACT']}, "
          f"Fitted={fitted_values[i]:.4f}, "
          f"Residual={residuals[i]:.4f}")

Fitted Values and Residuals for each observation:
   Student  GPA  ACT  Fitted_GPA  Residuals
0        1  2.8   21      2.7143     0.0857
1        2  3.4   24      3.0209     0.3791
2        3  3.0   26      3.2253    -0.2253
3        4  3.5   27      3.3275     0.1725
4        5  3.6   29      3.5319     0.0681
5        6  3.0   25      3.1231    -0.1231
6        7  2.7   25      3.1231    -0.4231
7        8  3.7   30      3.6341     0.0659

Sum of residuals: -0.0000000000
This should be approximately zero (subject to rounding error)

Detailed breakdown:
Student 1: GPA=2.8, ACT=21.0, Fitted=2.7143, Residual=0.0857
Student 2: GPA=3.4, ACT=24.0, Fitted=3.0209, Residual=0.3791
Student 3: GPA=3.0, ACT=26.0, Fitted=3.2253, Residual=-0.2253
Student 4: GPA=3.5, ACT=27.0, Fitted=3.3275, Residual=0.1725
Student 5: GPA=3.6, ACT=29.0, Fitted=3.5319, Residual=0.0681
Student 6: GPA=3.0, ACT=25.0, Fitted=3.1231, Residual=-0.1231
Student 7: GPA=2.7, ACT=25.0, Fitted=3.1231, Residual=-0.4231
Student 

### Part (iii): Prediction for ACT = 20

In [57]:
# Part (iii): Predict GPA when ACT = 20
predicted_gpa_act20 = results_gpa.predict([1, 20])[0]
print(f"\nPredicted GPA when ACT = 20: {predicted_gpa_act20:.4f}")



Predicted GPA when ACT = 20: 2.6121


### Part (iv): Explained Variation

In [64]:
# Get R-squared value
r_squared = results_gpa.rsquared

print(f"R-squared: {r_squared:.4f}")
print(f"Percentage of variation in GPA explained by ACT: {r_squared*100:.2f}%")
print(f"Percentage of variation in GPA NOT explained by ACT: {(1-r_squared)*100:.2f}%")


R-squared: 0.5774
Percentage of variation in GPA explained by ACT: 57.74%
Percentage of variation in GPA NOT explained by ACT: 42.26%


r^2 above 1/2 indicates high contribution in variation

---

## Problem 4: Birth Weight and Smoking

**Problem Statement:**

The data set BWGHT.RAW contains data on births to women in the United States. Two variables of interest are the dependent variable, infant birth weight in ounces (bwght), and an explanatory variable, average number of cigarettes the mother smoked per day during pregnancy (cigs). The following simple regression was estimated using data on n = 1,388 births:

bwght = 119.77 - 0.514 cigs

In [65]:
# Since bwght is mentioned in the column mapping, let's load the appropriate dataset
# The problem mentions BWGHT.RAW dataset
bwght = load_wooldridge_data('bwght')

print("BWGHT Dataset Overview:")
print(f"Shape: {bwght.shape}")
print(f"Columns: {list(bwght.columns)}")
print("\nFirst 5 rows:")
print(bwght.head())

# Check for the specific variables mentioned in the problem
if 'bwght' in bwght.columns and 'cigs' in bwght.columns:
    print(f"\nBirth weight (bwght) summary:")
    print(bwght['bwght'].describe())
    print(f"\nCigarettes (cigs) summary:")
    print(bwght['cigs'].describe())
else:
    print(f"\nAvailable columns that might be relevant:")
    relevant_cols = [col for col in bwght.columns if any(term in col.lower() for term in ['weight', 'birth', 'cig', 'smoke'])]
    print(relevant_cols)

CSV file not found for bwght. Downloading...
Successfully downloaded and converted to: d:\Users\achyu\git\learnStats\Econometrics\Practice\notebooks\..\data\bwght.csv
Loaded bwght from d:\Users\achyu\git\learnStats\Econometrics\Practice\notebooks\..\data\bwght.csv
Shape: (1388, 14)
BWGHT Dataset Overview:
Shape: (1388, 14)
Columns: ['faminc', 'cigtax', 'cigprice', 'bwght', 'fatheduc', 'motheduc', 'parity', 'male', 'white', 'cigs', 'lbwght', 'bwghtlbs', 'packs', 'lfaminc']

First 5 rows:
   faminc  cigtax  cigprice  bwght  fatheduc  motheduc  parity   male  white  \
0    13.5    16.5     122.3  109.0      12.0      12.0     1.0   True   True   
1     7.5    16.5     122.3  133.0       6.0      12.0     2.0   True  False   
2     0.5    16.5     122.3  129.0       NaN      12.0     2.0  False  False   
3    15.5    16.5     122.3  126.0      12.0      12.0     2.0   True  False   
4    27.5    16.5     122.3  134.0      14.0      12.0     2.0   True   True   

   cigs  lbwght  bwghtlbs  

### Given Regression Results

In [66]:
# Given regression: bwght = 119.77 - 0.514 cigs
# Sample size: n = 1,388

beta_0 = 119.77  # Intercept
beta_1 = -0.514  # Slope coefficient on cigs
n = 1388         # Sample size

print(f"Estimated regression: bwght = {beta_0} + {beta_1} * cigs")
print(f"Sample size: {n}")

Estimated regression: bwght = 119.77 + -0.514 * cigs
Sample size: 1388


### Part (i): Predictions for Different Smoking Levels
(i) What is the predicted birth weight when cigs = 0? What about when cigs = 20 (one pack per day)? Comment on the difference.

In [68]:
''' Part (i): Predict birth weight for cigs = 0 and cigs = 20 '''
# cigs_0 = 0
predicted_bwght_0 = beta_0 + beta_1 * 0
# cigs_20 = 20
predicted_bwght_20 = beta_0 + beta_1 * 20
print(f"Predicted birth weight when cigs = 0: {predicted_bwght_0:.2f} ounces")
print(f"Predicted birth weight when cigs = 20: {predicted_bwght_20:.2f} ounces")
print(f"Difference in predicted birth weight (cigs=0 vs cigs=20): {predicted_bwght_0 - predicted_bwght_20:.2f} ounces") 


Predicted birth weight when cigs = 0: 119.77 ounces
Predicted birth weight when cigs = 20: 109.49 ounces
Difference in predicted birth weight (cigs=0 vs cigs=20): 10.28 ounces


We see that the birthweight of babies who have been born to those who smoke more were lower 

### Part (ii): Causality Discussion
(ii) Does this simple regression necessarily capture a causal relationship between the child's birth weight and the mother's smoking habits? Explain.

The regression model can possibly capture a causal relationship as we notice that elevated cigarette smoking is correlated with lower infant birthrates. While the regression shows a strong negative correlation between smoking and birthweight, establishing causality requires controlling for confounding factors such as maternal education, income, age, and other health behaviors that might influence both smoking decisions and birth outcomes.

### Part (iii): Reverse Prediction
(iii) To predict a birth weight of 125 ounces, what would cigs have to be? Comment.

In [69]:
# Part (iii): Find cigs needed to predict bwght = 125 ounces
target_bwght = 125
# Rearranging the regression equation to solve for cigs
cigs_needed = (target_bwght - beta_0) / beta_1
print(f"Cigarettes needed to predict birth weight of {target_bwght} ounces: {cigs_needed:.2f} cigarettes")

Cigarettes needed to predict birth weight of 125 ounces: -10.18 cigarettes


This does not make sense as one cannot smoke negative cigarettes

### Part (iv): Sample Composition Analysis
(iv) The proportion of women in the sample who do not smoke while pregnant is about .85. Does this help reconcile your finding from part (iii)?

In [78]:
# Part (iv): Sample composition analysis

# First, let's examine the distribution of smoking in the sample
print("Smoking distribution in the sample:")
print(f"Total observations: {len(bwght)}")

# Count non-smokers (cigs = 0) and smokers (cigs > 0)
non_smokers = bwght[bwght['cigs'] == 0]
smokers = bwght[bwght['cigs'] > 0]

print(f"Non-smokers (cigs = 0): {len(non_smokers)} ({len(non_smokers)/len(bwght)*100:.1f}%)")
print(f"Smokers (cigs > 0): {len(smokers)} ({len(smokers)/len(bwght)*100:.1f}%)")

# Calculate average birth weights for each group
avg_bwght_nonsmokers = non_smokers['bwght'].mean()
avg_bwght_smokers = smokers['bwght'].mean()
std_bwght_nonsmokers = non_smokers['bwght'].std(ddof=1)

print(f"\nAverage birth weights:")
print(f"Non-smokers: {avg_bwght_nonsmokers:.2f} ounces")
print(f"Smokers: {avg_bwght_smokers:.2f} ounces")
print(f"Difference: {avg_bwght_nonsmokers - avg_bwght_smokers:.2f} ounces")
print(f"Standard deviation for non-smokers: {std_bwght_nonsmokers:.2f} ounces")

# CONFIDENCE INTERVAL EXPLANATION
import scipy.stats as stats
n_nonsmokers = len(non_smokers)
standard_error = std_bwght_nonsmokers / np.sqrt(n_nonsmokers)

print(f"\n--- CONFIDENCE INTERVAL EXPLANATION ---")
print(f"Sample size of non-smokers: {n_nonsmokers}")
print(f"Standard deviation of birth weights: {std_bwght_nonsmokers:.2f} ounces")
print(f"Standard error of the MEAN: {standard_error:.4f} ounces")
print(f"Why is standard error so small? Because SE = σ/√n = {std_bwght_nonsmokers:.2f}/√{n_nonsmokers} = {standard_error:.4f}")

# Calculate confidence intervals
confidence_level_95 = 0.95
confidence_level_90 = 0.90
degrees_freedom = n_nonsmokers - 1

t_critical_95 = stats.t.ppf((1 + confidence_level_95)/2, degrees_freedom)
t_critical_90 = stats.t.ppf((1 + confidence_level_90)/2, degrees_freedom)

margin_error_95 = t_critical_95 * standard_error
margin_error_90 = t_critical_90 * standard_error

ci_95_lower = avg_bwght_nonsmokers - margin_error_95
ci_95_upper = avg_bwght_nonsmokers + margin_error_95
ci_90_lower = avg_bwght_nonsmokers - margin_error_90
ci_90_upper = avg_bwght_nonsmokers + margin_error_90

print(f"\n95% confidence interval for AVERAGE birth weight: ({ci_95_lower:.2f}, {ci_95_upper:.2f}) ounces")
print(f"90% confidence interval for AVERAGE birth weight: ({ci_90_lower:.2f}, {ci_90_upper:.2f}) ounces")

Smoking distribution in the sample:
Total observations: 1388
Non-smokers (cigs = 0): 1176 (84.7%)
Smokers (cigs > 0): 212 (15.3%)

Average birth weights:
Non-smokers: 120.06 ounces
Smokers: 111.15 ounces
Difference: 8.91 ounces
Standard deviation for non-smokers: 20.27 ounces

--- CONFIDENCE INTERVAL EXPLANATION ---
Sample size of non-smokers: 1176
Standard deviation of birth weights: 20.27 ounces
Standard error of the MEAN: 0.5910 ounces
Why is standard error so small? Because SE = σ/√n = 20.27/√1176 = 0.5910

95% confidence interval for AVERAGE birth weight: (118.90, 121.22) ounces
90% confidence interval for AVERAGE birth weight: (119.09, 121.03) ounces


Given the standard error of the non smoking population, we can see with the confidence intervals at both 95% confidence and 90% confidence that observing a new sample birthweight of 125 ounces is quite unlikely, so the estimate that negative cigarettes would be expected to see such a birthweight actually helps reconcile the previous response; it would seem one would need negative cigarettes to see such a high birthweight.

---

## Notes and Key Takeaways

### Important Concepts from Chapter 2:

1. **Simple Linear Regression**: The foundation of econometric analysis using OLS estimation
2. **Interpretation of Coefficients**: Understanding intercepts, slopes, and their economic meaning
3. **Functional Forms**: Linear, log-linear, log-log models and their interpretations
4. **Goodness of Fit**: R-squared as a measure of explanatory power
5. **Causality vs Correlation**: Understanding the limitations of regression analysis

### Key Formulas:

- **Simple Regression Model**: y = β₀ + β₁x + u
- **OLS Estimators**: β̂₁ = Σ(xᵢ-x̄)(yᵢ-ȳ)/Σ(xᵢ-x̄)², β̂₀ = ȳ - β̂₁x̄
- **R-squared**: R² = ESS/TSS = 1 - SSR/TSS
- **Log-linear Interpretation**: %Δy ≈ (100×β₁)Δx for small changes
- **Log-log Interpretation**: %Δy ≈ β₁×(%Δx) (elasticity)

---

## References

- Wooldridge, J. M. (2013). *Introductory Econometrics: A Modern Approach* (5th ed.). Cengage Learning.
- Dataset source: https://faculty.utrgv.edu/diego.escobari/teaching/Datasets.html
- Papke, L. E. (1995). "Participation in and Contributions to 401(k) Pension Plans." *Journal of Human Resources*.
- Biddle, J. E. and D. S. Hamermesh (1990). "Sleep and the Allocation of Time." *Journal of Political Economy*.
- Franses, P. H. and R. Paap (2001). *Quantitative Models in Marketing Research*. Cambridge University Press.